**CS7643 Final Project: Comparing CNN and
Transformer Representations for Low-Data
Land-Use Classification**

**1. Google Drive & Kaggle Setup in Colab**

In [2]:
import os
import zipfile

# Define paths
GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'ColabNotebooks/cs7643/FinalProject'
PROJECT_ROOT = os.path.join('/content', 'drive', 'MyDrive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
data_dir = os.path.join(PROJECT_ROOT, 'data')

os.makedirs(data_dir, exist_ok=True)

# Move the downloaded zip into your project folder if it isn't already there
zip_src = '/content/data/eurosat-rgb-dataset.zip'
zip_dest = os.path.join(data_dir, 'eurosat-rgb-dataset.zip')

if os.path.exists(zip_src) and not os.path.exists(zip_dest):
    import shutil
    shutil.move(zip_src, zip_dest)
    print(f"Moved zip to: {zip_dest}")

# Unzip the dataset inside your project folder
extracted_path = os.path.join(data_dir, 'eurosat')
if not os.path.exists(extracted_path):
    with zipfile.ZipFile(zip_dest, 'r') as zip_ref:
        zip_ref.extractall(extracted_path)
    print("Dataset extracted successfully!")
else:
    print("Dataset already extracted.")

Moved zip to: /content/drive/MyDrive/ColabNotebooks/cs7643/FinalProject/data/eurosat-rgb-dataset.zip
Dataset extracted successfully!


**2. Split Base and Novel Classes for Few-Shot Learning**

In [3]:
import shutil

# Locate the extracted EuroSAT folder (adjust if subfolder name differs slightly)
# Usually Kaggle extracts into a folder like 'EuroSAT' or 'eurosat-rgb-dataset'
source_dir = os.path.join(extracted_path, 'EuroSAT')
if not os.path.exists(source_dir):
    # Fallback check if the inner folder has a different name
    subfolders = os.listdir(extracted_path)
    source_dir = os.path.join(extracted_path, subfolders[0])

split_root = os.path.join(PROJECT_ROOT, 'eurosat_split')
base_dest = os.path.join(split_root, 'base')
novel_dest = os.path.join(split_root, 'novel')

base_classes = ['AnnualCrop', 'Forest', 'Highway', 'Industrial', 'Residential', 'SeaLake']
novel_classes = ['HerbaceousVegetation', 'Pasture', 'PermanentCrop', 'River']

os.makedirs(base_dest, exist_ok=True)
os.makedirs(novel_dest, exist_ok=True)

for class_name in base_classes:
    src = os.path.join(source_dir, class_name)
    dst = os.path.join(base_dest, class_name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)

for class_name in novel_classes:
    src = os.path.join(source_dir, class_name)
    dst = os.path.join(novel_dest, class_name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)

print(f"Base and novel splits successfully created at: {split_root}")

Base and novel splits successfully created at: /content/drive/MyDrive/ColabNotebooks/cs7643/FinalProject/eurosat_split


**3. Base Class Data Split for Training**

In [4]:
import os
import random
import shutil

# Paths inside your shared Google Drive project folder
PROJECT_ROOT = '/content/drive/MyDrive/ColabNotebooks/cs7643/FinalProject'
base_src = os.path.join(PROJECT_ROOT, 'eurosat_split', 'base')
split_output = os.path.join(PROJECT_ROOT, 'base_splits')

# Ratios for your split
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
# Test ratio will be the remainder (~0.15)

random.seed(42)  # Ensures everyone gets the exact same split!

for class_name in os.listdir(base_src):
    class_path = os.path.join(base_src, class_name)
    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * TRAIN_RATIO)
    n_val = int(n_total * VAL_RATIO)

    splits = {
        'train': images[:n_train],
        'val': images[n_train:n_train + n_val],
        'test': images[n_train + n_val:]
    }

    for split_name, split_images in splits.items():
        dest_dir = os.path.join(split_output, split_name, class_name)
        os.makedirs(dest_dir, exist_ok=True)
        for img in split_images:
            src_img = os.path.join(class_path, img)
            dst_img = os.path.join(dest_dir, img)
            if not os.path.exists(dst_img):
                shutil.copy(src_img, dst_img)

print("Base classes successfully split into train, val, and test sets for the whole team!")

Base classes successfully split into train, val, and test sets for the whole team!


In [7]:
train_dir = os.path.join(PROJECT_ROOT, 'base_splits', 'train')

total_train_images = 0
for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        total_train_images += count
        print(f"  - {class_name}: {count} images")

print(f"\nTotal images in training split: {total_train_images}")

  - AnnualCrop: 2100 images
  - Forest: 2100 images
  - Highway: 1750 images
  - Industrial: 1750 images
  - Residential: 2100 images
  - SeaLake: 2100 images

Total images in training split: 11900


In [9]:
test_dir = os.path.join(PROJECT_ROOT, 'base_splits', 'test')

total_test_images = 0
for class_name in os.listdir(test_dir):
    class_path = os.path.join(test_dir, class_name)
    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        total_test_images += count
        print(f"  - {class_name}: {count} images")

print(f"\nTotal images in test split: {total_test_images}")

  - AnnualCrop: 450 images
  - Forest: 450 images
  - Highway: 375 images
  - Industrial: 375 images
  - Residential: 450 images
  - SeaLake: 450 images

Total images in test split: 2550


**4. Data Loaders and Data Transformation for Base Classes (need to share with team)**

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

PROJECT_ROOT = '/content/drive/MyDrive/ColabNotebooks/cs7643/FinalProject'
base_splits_dir = os.path.join(PROJECT_ROOT, 'base_splits')

# Standard ImageNet preprocessing for pretrained models
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {
    x: datasets.ImageFolder(os.path.join(base_splits_dir, x), data_transforms[x])
    for x in ['train', 'val']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2)
    for x in ['train', 'val']
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
print(f"Base Classes: {class_names}")

Base Classes: ['AnnualCrop', 'Forest', 'Highway', 'Industrial', 'Residential', 'SeaLake']


**5.Initialize ResNet-18 and Modify the Classifier**

Load a pretrained ResNet-18 and swap out the final fully connected layer to match 6 base classes

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load pretrained ResNet-18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze early layers to test pure feature extraction,
# or leave them unfrozen for fine-tuning. Let's fine-tune the whole network lightly:
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 197MB/s]


In [14]:
import time
start_time = time.time()

num_epochs = 3

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

print("Sanity check training complete!")
end_time = time.time()
print("Time taken:", (end_time - start_time)/60)

Epoch 1/3
----------
train Loss: 0.2742 Acc: 0.9128
val Loss: 0.2170 Acc: 0.9455
Epoch 2/3
----------
train Loss: 0.1455 Acc: 0.9533
val Loss: 0.0549 Acc: 0.9867
Epoch 3/3
----------
train Loss: 0.1065 Acc: 0.9656
val Loss: 0.0808 Acc: 0.9725
Sanity check training complete!


**6. Few-Shot Evaluation (This also needs to be shared with team)**

In [16]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.linear_model import RidgeClassifier

# 1. Setup paths and device
PROJECT_ROOT = '/content/drive/MyDrive/ColabNotebooks/cs7643/FinalProject'
novel_dir = os.path.join(PROJECT_ROOT, 'eurosat_split', 'novel')
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 2. Load your trained ResNet-18 and convert it to a feature extractor
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 6) # Match your base class count
# Load your saved weights if running in a fresh session:
# model.load_state_dict(torch.load(os.path.join(PROJECT_ROOT, 'resnet18_base.pth')))

model.to(device)
model.eval()

# Strip the final classification layer to get 512-dim feature embeddings
feature_extractor = nn.Sequential(*(list(model.children())[:-1]))
feature_extractor.eval()

# 3. Transform for novel dataset (Validation transform only)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

novel_dataset = datasets.ImageFolder(novel_dir, transform=transform)
novel_classes = novel_dataset.classes
print(f"Novel Classes for Evaluation: {novel_classes}")

# Group indices by class for easy K-shot sampling
class_to_indices = {i: [] for i in range(len(novel_classes))}
for idx, (_, label) in enumerate(novel_dataset):
    class_to_indices[label].append(idx)

# 4. Updated Flexible Few-shot Evaluation Function
def evaluate_few_shot(k_shot, method='centroid_euclidean', num_episodes=5):
    accuracies = []

    for episode in range(num_episodes):
        support_indices = []
        query_indices = []

        # Sample K support images and remaining query images per class
        for c_idx, indices in class_to_indices.items():
            sampled = random.sample(indices, k_shot + 200) # K for support, 200 for query test
            support_indices.extend(sampled[:k_shot])
            query_indices.extend(sampled[k_shot:])

        support_loader = DataLoader(Subset(novel_dataset, support_indices), batch_size=16, shuffle=False)
        query_loader = DataLoader(Subset(novel_dataset, query_indices), batch_size=32, shuffle=False)

        # Extract support features and labels
        support_feats, support_labels = [], []
        with torch.no_grad():
            for inputs, labels in support_loader:
                feats = feature_extractor(inputs.to(device)).squeeze(-1).squeeze(-1)
                support_feats.append(feats.cpu())
                support_labels.append(labels)
        support_feats = torch.cat(support_feats)
        support_labels = torch.cat(support_labels)

        # --- METHOD 1: Ridge Regression Classifier ---
        if method == 'ridge':
            clf = RidgeClassifier(alpha=1.0)
            clf.fit(support_feats.numpy(), support_labels.numpy())

            correct = 0
            total = 0
            with torch.no_grad():
                for inputs, labels in query_loader:
                    inputs = inputs.to(device)
                    labels = labels.to(device)
                    feats = feature_extractor(inputs).squeeze(-1).squeeze(-1)

                    preds = clf.predict(feats.cpu().numpy())
                    preds_tensor = torch.tensor(preds, device=device)

                    correct += (preds_tensor == labels).sum().item()
                    total += labels.size(0)
            accuracies.append(correct / total)

        # --- METHOD 2 & 3: Prototypical (Euclidean vs Cosine) ---
        else:
            if method == 'centroid_cosine':
                support_feats = F.normalize(support_feats, p=2, dim=1)

            # Compute prototypes
            prototypes = []
            for c_idx in range(len(novel_classes)):
                class_mask = (support_labels == c_idx)
                proto = support_feats[class_mask].mean(dim=0)
                prototypes.append(proto)
            prototypes = torch.stack(prototypes).to(device)

            if method == 'centroid_cosine':
                prototypes = F.normalize(prototypes, p=2, dim=1)

            # Evaluate on query set
            correct = 0
            total = 0
            with torch.no_grad():
                for inputs, labels in query_loader:
                    inputs = inputs.to(device)
                    labels = labels.to(device)
                    feats = feature_extractor(inputs).squeeze(-1).squeeze(-1)

                    if method == 'centroid_cosine':
                        feats = F.normalize(feats, p=2, dim=1)
                        # Cosine similarity (higher is closer)
                        sims = torch.mm(feats, prototypes.t())
                        preds = torch.argmax(sims, dim=1)
                    else:
                        # Euclidean distance (lower is closer)
                        dists = torch.cdist(feats, prototypes, p=2)
                        preds = torch.argmin(dists, dim=1)

                    correct += (preds == labels).sum().item()
                    total += labels.size(0)

            accuracies.append(correct / total)

    mean_acc = np.mean(accuracies) * 100
    std_acc = np.std(accuracies) * 100
    print(f"[{k_shot}-shot | {method}] Accuracy: {mean_acc:.2f}% (± {std_acc:.2f}%)")

# 5. Compare all three methods back-to-back
print("\n--- Comparing Few-Shot Methods ---")
for k in [1, 5, 10]:
    for m in ['centroid_euclidean', 'centroid_cosine', 'ridge']:
        evaluate_few_shot(k_shot=k, method=m, num_episodes=5)

Novel Classes for Evaluation: ['HerbaceousVegetation', 'Pasture', 'PermanentCrop', 'River']

--- Comparing Few-Shot Methods ---
[1-shot | centroid_euclidean] Accuracy: 32.03% (± 4.05%)
[1-shot | centroid_cosine] Accuracy: 38.43% (± 9.03%)
[1-shot | ridge] Accuracy: 33.77% (± 6.93%)
[5-shot | centroid_euclidean] Accuracy: 37.50% (± 6.25%)
[5-shot | centroid_cosine] Accuracy: 43.42% (± 6.24%)
[5-shot | ridge] Accuracy: 60.05% (± 4.64%)
[10-shot | centroid_euclidean] Accuracy: 40.80% (± 5.70%)
[10-shot | centroid_cosine] Accuracy: 45.73% (± 2.58%)
[10-shot | ridge] Accuracy: 58.62% (± 6.79%)
